# D7: Within-Sentence Token-Pair Swap + Exponent Analysis

Swaps adjacent token pairs *within* sentences, preserving sentence order.
Compares: intact > D6 (sentence swap) > D7 (token swap) > D4 (full reverse)?
Or: D7 > D6? Empirically interesting either way.

Also fits power law exponents for all conditions.
No shuffles — raw marginals. ~15 min per corpus per new condition.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time, shutil
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_D6')  # reuse D6 directory
BASE.mkdir(parents=True, exist_ok=True)
FORMAL = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
RAW_PILOT = Path('/content/drive/MyDrive/LRTIA/Results/Exp1_disruption_raw')
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10

# Copy intact raw if not present
for cn in CORPORA:
    dst = BASE / f'llama_{cn}_intact_raw.json'
    if not dst.exists():
        # Try to build from formal corrected results
        src = FORMAL / f'llama_{cn}_intact.json'
        if src.exists():
            with open(src) as f: formal = json.load(f)
            raw = []
            for r in formal:
                ppls = r['ordered_ppl']
                dists = list(range(1, len(ppls)))
                marg = [ppls[d-1] - ppls[d] for d in dists]
                raw.append({'distances': dists, 'ppls': ppls, 'marginals': marg,
                            'doc_id': r.get('doc_id',''), 'target_frac': r.get('target_frac',0)})
            with open(dst, 'w') as f: json.dump(raw, f)
            print(f'Built {cn} intact raw from formal ({len(raw)})')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup done')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === Disruption functions ===

SENT_BOUNDS = set('. ? ! 。 ？ ！ ؟'.split())

def segment_sentences(token_ids, tokenizer):
    units = []
    start = 0
    for i, tid in enumerate(token_ids):
        s = tokenizer.decode([tid]).strip()
        if any(ch in SENT_BOUNDS for ch in s):
            units.append((start, i + 1))
            start = i + 1
    if start < len(token_ids):
        units.append((start, len(token_ids)))
    return units

def swap_adjacent_pairs(tokens):
    """Swap adjacent token pairs: [a,b,c,d,e] -> [b,a,d,c,e]"""
    result = list(tokens)
    i = 0
    while i < len(result) - 1:
        result[i], result[i+1] = result[i+1], result[i]
        i += 2
    return result

def d7_within_sentence_token_swap(ctx, tokenizer):
    """Swap adjacent token pairs within each sentence unit.
    Preserves sentence order and boundaries."""
    units = segment_sentences(ctx, tokenizer)
    result = []
    for start, end in units:
        sentence_tokens = ctx[start:end]
        swapped = swap_adjacent_pairs(sentence_tokens)
        result.extend(swapped)
    return result, len(units)

@torch.no_grad()
def ppl_only(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids)
    logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i+1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf')
    return math.exp(nll / cnt)

def compute_raw_curves(cond_ctx, tgt):
    mc = len(cond_ctx)
    ppls = []
    for c in range(mc + 1):
        pfx = cond_ctx[-c:] if c > 0 else []
        ppls.append(ppl_only(pfx, tgt))
    dists = list(range(1, mc + 1))
    marginals = [ppls[d-1] - ppls[d] for d in dists]
    return {'distances': dists, 'ppls': ppls, 'marginals': marginals}

# Test D7
with open(CORPORA['wiki_zh']) as f:
    doc = json.loads(f.readline())
ids = tokenizer.encode(doc['text'], add_special_tokens=False)
ctx = ids[len(ids)//2 - C : len(ids)//2]

d7, nu = d7_within_sentence_token_swap(ctx, tokenizer)
assert len(d7) == len(ctx), f'Length: {len(d7)} vs {len(ctx)}'
assert Counter(d7) == Counter(ctx), 'Multiset changed'
print(f'D7 test: {nu} sentence units, length ok, multiset ok')
# Show a sample
units = segment_sentences(ctx, tokenizer)
s, e = units[0]
orig = tokenizer.decode(ctx[s:e])
swpd = tokenizer.decode(d7[s:e])
print(f'  Original: "{orig[:80]}"')
print(f'  D7:       "{swpd[:80]}"')
print('Functions ready')

In [ ]:
# === Run D7 (D6 + intact already cached) ===

for cn, cp in CORPORA.items():
    print(f'\n{"="*60}')
    print(cn)
    print(f'{"="*60}')
    
    cache = BASE / f'llama_{cn}_D7_raw.json'
    if cache.exists():
        with open(cache) as f: n = len(json.load(f))
        print(f'  D7: cached ({n})'); continue
    
    docs = []
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    t0 = time.time()
    results = []
    
    for doc in tqdm(docs, desc=f'{cn}/D7'):
        fids = tokenizer.encode(doc['text'], add_special_tokens=False)
        n = len(fids)
        for frac in TARGET_FRACS:
            ts = int(n * frac)
            te = min(ts + TARGET_LEN, n)
            if ts < MIN_BEFORE or te - ts < 5: continue
            ctx = fids[ts-C:ts]
            tgt = fids[ts:te]
            
            d7_ctx, nu = d7_within_sentence_token_swap(ctx, tokenizer)
            assert len(d7_ctx) == C and Counter(d7_ctx) == Counter(ctx)
            
            r = compute_raw_curves(d7_ctx, tgt)
            r['doc_id'] = doc.get('doc_id', '')
            r['target_frac'] = frac
            r['n_units'] = nu
            results.append(r)
    
    with open(cache, 'w') as f: json.dump(results, f)
    elapsed = time.time() - t0
    print(f'  D7: {len(results)} results in {elapsed/60:.1f} min')
    if results:
        print(f'  Mean raw marginal: {np.mean([np.mean(r["marginals"]) for r in results]):.6f}')

In [ ]:
# === Full comparison table + power law exponents ===
import matplotlib.pyplot as plt

bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p
    return None, None, None

# Collect all conditions
ALL_CONDS = [
    ('intact',  'llama_{cn}_intact_raw.json',  BASE,       'marginals', 'blue'),
    ('D6 sent', 'llama_{cn}_D6_raw.json',      BASE,       'marginals', 'orange'),
    ('D7 tok',  'llama_{cn}_D7_raw.json',      BASE,       'marginals', 'purple'),
    ('D4 rev',  'llama_{cn}_D4.json',          FORMAL,     'delta_ppl', 'green'),
]

print(f'{"Corpus":<12} {"Cond":<10} {"TotalΔ":>8} {"NearΔ":>8} {"MidΔ":>8} {"FarΔ":>8} {"α":>8} {"r":>8}')
print('-' * 75)

for cn in CORPORA:
    for cond_name, fname_tmpl, src_base, key, color in ALL_CONDS:
        fname = fname_tmpl.format(cn=cn)
        cp = src_base / fname
        if not cp.exists():
            print(f'{cn:<12} {cond_name:<10} {"—":>8}'); continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        
        curve = np.mean([r[key] for r in results], axis=0)
        total = np.mean(curve)
        near = np.mean(curve[:30])
        mid = np.mean(curve[30:70])
        far = np.mean(curve[70:])
        
        alpha, r_val, p_val = fit_power_law(np.array(curve))
        a_str = f'{alpha:.3f}' if alpha is not None else '—'
        r_str = f'{r_val:.3f}' if r_val is not None else '—'
        
        print(f'{cn:<12} {cond_name:<10} {total:>8.4f} {near:>8.4f} {mid:>8.4f} {far:>8.4f} {a_str:>8} {r_str:>8}')
    print()

In [ ]:
# === Plots: PPL curves + raw marginals for all 4 conditions ===

fig, axes = plt.subplots(2, len(CORPORA), figsize=(7*len(CORPORA), 10))
if len(CORPORA) == 1: axes = axes.reshape(-1, 1)

for idx, cn in enumerate(CORPORA):
    # PPL curves
    ax = axes[0, idx]
    for cond_name, fname_tmpl, src_base, key, color in ALL_CONDS:
        fname = fname_tmpl.format(cn=cn)
        cp = src_base / fname
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        if 'ppls' in results[0]:
            ppl = np.mean([r['ppls'] for r in results], axis=0)
            ax.plot(range(len(ppl)), ppl, color=color, linewidth=2, label=cond_name)
        elif 'ordered_ppl' in results[0]:
            ppl = np.mean([r['ordered_ppl'] for r in results], axis=0)
            ax.plot(range(len(ppl)), ppl, color=color, linewidth=2, label=cond_name)
    ax.set_title(f'{cn} — PPL Curves', fontweight='bold')
    ax.set_xlabel('Context length c')
    ax.set_ylabel('Perplexity')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)
    
    # Raw marginals
    ax = axes[1, idx]
    for cond_name, fname_tmpl, src_base, key, color in ALL_CONDS:
        fname = fname_tmpl.format(cn=cn)
        cp = src_base / fname
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        curve = np.mean([r[key] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth)+1), smooth, color=color, linewidth=2, label=cond_name)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(f'{cn} — Marginals', fontweight='bold')
    ax.set_xlabel('Distance d')
    ax.set_ylabel('Marginal (PPL drop)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

plt.suptitle('Disruption Hierarchy: intact > D6 (sent swap) > D7 (tok swap) > D4 (reverse)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D7_hierarchy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done')